In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgbm

In [3]:
# 데이터 분할 및 전처리
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 특성 선택
from sklearn.feature_selection import SequentialFeatureSelector

# 하이퍼파라미터 튜닝
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import precision_recall_curve
# 모델 평가 지표
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

# 데이터 시각화
import seaborn as sns

### 1. 데이터 로드 및 탐색

#### 1.1 데이터 로드

In [5]:
df_trn = pd.read_csv('train.csv')
df_tst = pd.read_csv('test.csv')

### 2. 데이터 전처리

#### 2.1 범주형 데이터 인코딩

In [14]:
# 범주형 데이터를 라벨 인코딩
cat = ['job','marital','contact','poutcome','education','default','housing','loan','month']
label_encoder = {}
for col in cat:
    le = LabelEncoder()
    df_trn[col] = le.fit_transform(df_trn[col])
    df_tst[col] = le.transform(df_tst[col])
    label_encoder[col] = le

#### 2.2 파생 변수 생성

In [15]:
# 계절을 반환하는 함수 정의
def get_season(month):
    if 3 <= month <= 5:
        return 'spring'
    elif 6 <= month <= 8:
        return 'summer'
    elif 9 <= month <= 11:
        return 'autumn'
    else:
        return 'winter'

In [16]:
# 데이터 전처리 함수 정의
def preprocess_data(df_trn, df_tst):

    le = LabelEncoder()
    
    # pdays 를 갖고 연락여부로 이진분류한 새로운 칼럼 생성
    df_trn['pdays_contacted'] = (df_trn['pdays'] != -1).astype(int)
    df_tst['pdays_contacted'] = (df_tst['pdays'] != -1).astype(int)
    
    # 흠....
    df_trn['duration_per_campaign'] = df_trn['duration'] / (df_trn['campaign'] + 1)
    df_tst['duration_per_campaign'] = df_tst['duration'] / (df_tst['campaign'] + 1)
    
    # day와 month를 통합하여 연속적인 시간 정보를 제공하기 위한 새로운 칼럼 생성
    df_trn['day_of_year'] = df_trn['day'] + (df_trn['month'] - 1) * 30
    df_tst['day_of_year'] = df_tst['day'] + (df_tst['month'] - 1) * 30
    
    # 계별 특성을 학습할 수 있도록 새로운 칼럼 생성
    df_trn['season'] = df_trn['month'].apply(get_season)
    df_tst['season'] = df_tst['month'].apply(get_season)
    # 범주형인 season을 라벨인코딩 수행
    df_trn['season'] = le.fit_transform(df_trn['season'])
    df_tst['season'] = le.transform(df_tst['season'])
    
    return df_trn, df_tst

In [17]:
# 전처리 함수 적용 후 할당
df_trn_preprocessed, df_tst_preprocessed = preprocess_data(df_trn,df_tst)

#### 2.3 Feature와 Target 분리

In [18]:
total_cols = [x for x in df_trn.columns]

id_cols = ['ID']
y_cols = ['label']
x_cols = [x for x in total_cols if x not in id_cols+y_cols]

In [19]:
# feature와 target분리
df_trn_x = df_trn_preprocessed[x_cols]
df_trn_y = df_trn_preprocessed[y_cols].values.ravel()
df_tst_x = df_tst_preprocessed[x_cols]

In [20]:
df_trn_x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31647 entries, 0 to 31646
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   age                    31647 non-null  int64  
 1   job                    31647 non-null  int64  
 2   marital                31647 non-null  int64  
 3   education              31647 non-null  int64  
 4   default                31647 non-null  int64  
 5   balance                31647 non-null  int64  
 6   housing                31647 non-null  int64  
 7   loan                   31647 non-null  int64  
 8   contact                31647 non-null  int64  
 9   day                    31647 non-null  int64  
 10  month                  31647 non-null  int64  
 11  duration               31647 non-null  int64  
 12  campaign               31647 non-null  int64  
 13  pdays                  31647 non-null  int64  
 14  previous               31647 non-null  int64  
 15  po

### 3. 모델 정의 및 학습

#### 3.1 검증 데이터 분리

In [ ]:
# 검증 데이터셋 분리
df_retrn_x, df_val_x, df_retrn_y, df_val_y = train_test_split(df_trn_x, df_trn_y, test_size=0.2, random_state=42, stratify=df_trn_y)

#### 3.2 초기 모델 정의 및 학습

In [22]:
# 초기 모델 정의
base_model = lgbm.LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced')
#학습
base_model.fit(df_retrn_x, df_retrn_y)

LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1)

#### 3.3 변수 중요도 분석

In [23]:
# Feature Importance 추출
importances = base_model.feature_importances_
features = df_retrn_x.columns

# 중요도를 데이터프레임으로 저장
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# 출력
print(importance_df)

                  Feature  Importance
18            day_of_year         588
11               duration         429
5                 balance         275
0                     age         273
17  duration_per_campaign         271
9                     day         214
13                  pdays         184
1                     job         126
8                 contact          93
10                  month          92
12               campaign          74
15               poutcome          71
3               education          67
6                 housing          63
14               previous          54
19                 season          48
2                 marital          41
7                    loan          28
4                 default           9
16        pdays_contacted           0


#### 3.4 변수 선택법

In [24]:
# 변수 선택 모델 정의
sfs = SequentialFeatureSelector(
    base_model,
    n_features_to_select='auto',
    direction='forward',
    scoring='f1',
    cv=5,
    n_jobs=-1
)
# 변수 선택 모델 학습하기
sfs.fit(df_retrn_x, df_retrn_y)

SequentialFeatureSelector(estimator=LGBMClassifier(class_weight='balanced',
                                                   random_state=42,
                                                   verbose=-1),
                          n_jobs=-1, scoring='f1')

In [25]:
# 선택된 변수 할당하기
selected_features = df_retrn_x.columns[sfs.get_support()].tolist()
selected_features = ['age', 'housing', 'contact', 'day', 'duration', 'campaign', 'pdays', 'poutcome', 'duration_per_campaign', 'day_of_year']
print(f"선택된 변수: {selected_features}")

선택된 변수: ['age', 'housing', 'contact', 'day', 'duration', 'campaign', 'pdays', 'poutcome', 'duration_per_campaign', 'day_of_year']


In [26]:
# 선택된 변수로 데이터 재구성
df_retrn_x_selected = df_retrn_x[selected_features]
df_val_x_selected = df_val_x[selected_features]
df_tst_x_selected = df_tst_x[selected_features]

### 4. 모델 최적화

#### 4.1 하이퍼파라미터 튜닝

In [27]:
# 파라미터 리스트 설정
param_random = {
    'learning_rate': [0.1,0.15,0.2],
    'n_estimators': [300,400,500],
    'max_depth': [3,4,5],
    'min_child_samples': [50, 100, 150],
    'subsample': [0.7,0.8,0.9],
    'colsample_bytree': [0.6,0.7,0.8],
    'class_weight': ['balanced']
}

In [28]:
# RandomizedSearchCV 정의 및 학습
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_random,
    n_iter=50,
    scoring='f1',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search.fit(df_retrn_x_selected, df_retrn_y)
best_model = random_search.best_estimator_
print(f"최적 하이퍼파라미터: {random_search.best_params_}")

최적 하이퍼파라미터: {'subsample': 0.8, 'n_estimators': 400, 'min_child_samples': 100, 'max_depth': 5, 'learning_rate': 0.15, 'colsample_bytree': 0.6, 'class_weight': 'balanced'}


In [29]:
best_model = lgbm.LGBMClassifier(random_state=42,
                                 verbose=-1,
                                 class_weight='balanced',
                                 subsample=0.8,
                                 n_estimators=400,
                                 max_depth=5,
                                 learning_rate=0.15,
                                 colsample_bytree=0.6,
                                 min_child_samples=100)
best_model.fit(df_retrn_x_selected, df_retrn_y)

LGBMClassifier(class_weight='balanced', colsample_bytree=0.6,
               learning_rate=0.15, max_depth=5, min_child_samples=100,
               n_estimators=400, random_state=42, subsample=0.8, verbose=-1)

#### 4.2 최적의 threshold 찾기

In [30]:
# 예측 확률값
y_prob = best_model.predict_proba(df_val_x_selected)[:, 1]  # 양성 클래스 확률

# Precision, Recall, Threshold 계산
precisions, recalls, thresholds = precision_recall_curve(df_val_y, y_prob)

# F1 Score 계산
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)

# 최적 Threshold 선택
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"Best Threshold for F1 Score: {best_threshold}")

# 최적 Threshold 기반 예측
y_pred = (y_prob >= best_threshold).astype(int)

# F1 Score 확인
print(f"Optimized F1 Score: {f1_score(df_val_y, y_pred)}")


Best Threshold for F1 Score: 0.7211056140719805
Optimized F1 Score: 0.6177156177156177


### 5. 모델 평가

In [38]:
# threshold 설정
thres=0.7211

In [39]:
# 훈련 데이터에 대한 성능 평가
y_retrn_pred_proba = best_model.predict_proba(df_retrn_x_selected)[:, 1]
y_retrn_pred_label = (y_retrn_pred_proba > thres).astype(int)

print(f"훈련 데이터 Accuracy: {accuracy_score(df_retrn_y, y_retrn_pred_label)}")
print(f"훈련 데이터 Precision: {precision_score(df_retrn_y, y_retrn_pred_label)}")
print(f"훈련 데이터 Recall: {recall_score(df_retrn_y, y_retrn_pred_label)}")
print(f"훈련 데이터 F1 Score: {f1_score(df_retrn_y, y_retrn_pred_label)}")

훈련 데이터 Accuracy: 0.929099024370976
훈련 데이터 Precision: 0.6480588683075361
훈련 데이터 Recall: 0.8622552329507089
훈련 데이터 F1 Score: 0.7399681297986382


검증 데이터로 성능 평가

In [40]:
# 검증 데이터에 대한 성능 평가
y_val_pred_proba = best_model.predict_proba(df_val_x_selected)[:, 1]
y_val_pred_label = (y_val_pred_proba > thres).astype(int)

print(f"검증 데이터 Accuracy: {accuracy_score(df_val_y, y_val_pred_label)}")
print(f"검증 데이터 Precision: {precision_score(df_val_y, y_val_pred_label)}")
print(f"검증 데이터 Recall: {recall_score(df_val_y, y_val_pred_label)}")
print(f"검증 데이터 F1 Score: {f1_score(df_val_y, y_val_pred_label)}")

검증 데이터 Accuracy: 0.9268562401263823
검증 데이터 Precision: 0.635386119257087
검증 데이터 Recall: 0.8783783783783784
검증 데이터 F1 Score: 0.737379466817924


### 6. 최종 모델 학습(전체 훈련 데이터)

In [41]:
df_trn_x_selected = df_trn_x[selected_features]
best_model.fit(df_trn_x_selected, df_trn_y)

LGBMClassifier(class_weight='balanced', colsample_bytree=0.6,
               learning_rate=0.15, max_depth=5, min_child_samples=100,
               n_estimators=400, random_state=42, subsample=0.8, verbose=-1)

### 7. 테스트 데이터에 대한 예측 후 저장하기

#### 7.1 테스트데이터에 대한 예측 수행

In [42]:
y_tst_pred_proba = best_model.predict_proba(df_tst_x_selected)[:,1]

In [43]:
y_tst_pred_label = (y_tst_pred_proba > thres).astype(int)

#### 7.2 예측결과 저장하기

In [45]:
df_submission = pd.read_csv('C:/Users/c0nc2/OneDrive/문서/카카오톡 받은 파일/submission_example.csv')

In [ ]:
df_submission.head()

,ID,label
0,test00001,0
1,test00002,0
2,test00003,0
3,test00004,0
4,test00005,0


In [47]:
df_submission['label'] = y_tst_pred_label

In [48]:
df_submission['label'].value_counts()

label
0    11541
1     2023
Name: count, dtype: int64

In [51]:
# 다하고 눌러라!!!!
filename = 'LightGBM_Ho_final-1.csv'
df_submission.to_csv(filename, index=False)